In [ ]:
import sys
sys.path.append("..")

from src.analytical.build_base_analitica import build_base_analitica



In [ ]:
import pandas as pd

dim_municipio = pd.read_parquet(
    "../data/gold/dimensions/dim_municipio/dim_municipio.parquet"
)

df = pd.read_parquet(
    "../data/gold/facts/fato_alfabetizacao_municipio/fato_alfabetizacao_municipio.parquet"
)

In [ ]:
df_analitica = build_base_analitica(
    df,
    dim_municipio
)

In [4]:
df_analitica.shape

(4611, 11)

In [5]:
df_analitica["id_municipio"].nunique()

4611

In [6]:
df_analitica["UF"].nunique()

26

In [7]:
df_analitica["target_atingiu_meta_2024"].value_counts(normalize=True)

target_atingiu_meta_2024
0    0.618955
1    0.381045
Name: proportion, dtype: float64

In [8]:
df_analitica.isna().sum()

id_municipio                 0
taxa_alfabetizacao_2023      0
taxa_presenca_2023           0
taxa_preenchimento_2023      0
alunos_avaliados_2023        0
alunos_alfabetizados_2023    0
alunos_presentes_2023        0
provas_preenchidas_2023      0
meta_2024                    0
target_atingiu_meta_2024     0
UF                           0
dtype: int64

### Estatísticas da base analítica

In [9]:
print("===== SHAPE =====")
print(df_analitica.shape)

print("\n===== COLUNAS =====")
print(df_analitica.columns.tolist())

print("\n===== TIPOS =====")
print(df_analitica.dtypes)

print("\n===== NULOS =====")
print(df_analitica.isna().sum())

print("\n===== MUNICÍPIOS =====")
print(df_analitica["id_municipio"].nunique())

print("\n===== UFs =====")
print(df_analitica["UF"].nunique())

===== SHAPE =====
(4611, 11)

===== COLUNAS =====
['id_municipio', 'taxa_alfabetizacao_2023', 'taxa_presenca_2023', 'taxa_preenchimento_2023', 'alunos_avaliados_2023', 'alunos_alfabetizados_2023', 'alunos_presentes_2023', 'provas_preenchidas_2023', 'meta_2024', 'target_atingiu_meta_2024', 'UF']

===== TIPOS =====
id_municipio                     str
taxa_alfabetizacao_2023      float64
taxa_presenca_2023           float64
taxa_preenchimento_2023      float64
alunos_avaliados_2023          int64
alunos_alfabetizados_2023      int64
alunos_presentes_2023          int64
provas_preenchidas_2023        int64
meta_2024                    float64
target_atingiu_meta_2024       int64
UF                               str
dtype: object

===== NULOS =====
id_municipio                 0
taxa_alfabetizacao_2023      0
taxa_presenca_2023           0
taxa_preenchimento_2023      0
alunos_avaliados_2023        0
alunos_alfabetizados_2023    0
alunos_presentes_2023        0
provas_preenchidas_2023     

In [10]:
df_analitica["target_atingiu_meta_2024"].value_counts(
    normalize=True
).mul(100).round(2)

target_atingiu_meta_2024
0    61.9
1    38.1
Name: proportion, dtype: float64

In [11]:
df_analitica.groupby(
    "target_atingiu_meta_2024"
).agg(
    municipios=("id_municipio", "count"),
    alfabetizacao_2023=("taxa_alfabetizacao_2023", "mean"),
    presenca_2023=("taxa_presenca_2023", "mean"),
    preenchimento_2023=("taxa_preenchimento_2023", "mean"),
    alunos_avaliados=("alunos_avaliados_2023", "mean")
)

,municipios,alfabetizacao_2023,presenca_2023,preenchimento_2023,alunos_avaliados
target_atingiu_meta_2024,,,,,
0,2854,54.735680,88.399846,88.392239,416.787316
1,1757,55.116949,92.065862,92.057479,195.154809


Parece que os municípios que atingiram a meta em 2024 já tinham um indicador melhor de alfabetização em 2023, além de indíce maiores de presença e preenchimento. Hipótese de evasão escolar em municípios que não atingiram a meta.

In [13]:
df_analitica.groupby("UF").agg(
    municipios=("id_municipio", "count"),
    taxa_alfabetizacao_2023=(
        "taxa_alfabetizacao_2023",
        "mean"
    ),
    taxa_presenca_2023=(
        "taxa_presenca_2023",
        "mean"
    ),
    taxa_preenchimento_2023=(
        "taxa_preenchimento_2023",
        "mean"
    ),
    percentual_atingiu_meta=(
        "target_atingiu_meta_2024",
        "mean"
    )
).sort_values(
    "municipios",
    ascending=False
)

,municipios,taxa_alfabetizacao_2023,taxa_presenca_2023,taxa_preenchimento_2023,percentual_atingiu_meta
UF,,,,,
MG,801,58.480762,91.944906,91.944906,0.626717
RS,421,64.573919,87.552660,87.552660,0.057007
PR,395,68.192456,89.254177,89.254177,0.184810
BA,394,33.354721,86.792234,86.792234,0.124365
GO,242,65.208306,89.672934,89.672934,0.636364
SC,238,58.129958,84.900714,84.900714,0.151261
PI,224,57.728348,95.537009,95.537009,0.544643
PB,220,49.488727,90.749273,90.749273,0.450000
MA,216,56.146991,91.182500,91.182500,0.416667


Observa-se que na base atual existem estados, como o caso de SP, que estão sub-representados em volume de municípios. Todavia, sabemos que existe bases auxiliares que podem ser usados para enriquecer a base de dados. 